# 🚢 Laboratorio: Regresión Logística con el caso Titanic

## 📖 La historia
La madrugada del **15 de abril de 1912** el Titanic chocó con un iceberg. De los **891 pasajeros** de los que tenemos registro, sobrevivió el **38%**. No fue al azar: la orden fue *"mujeres y niños primero"*, y los botes salvavidas estaban más cerca de los camarotes de primera clase.

Somos analistas de una aseguradora que quiere entender el riesgo en accidentes marítimos y nos preguntan:

> ### ❓ ¿Quién sobrevivió al Titanic, y podemos predecir la supervivencia de un pasajero a partir de sus características?

Esta es la misma idea de la sesión anterior, pero con un cambio clave: **ya no predecimos un número, predecimos una categoría**. En vez de "¿cuántas millas por galón?", ahora es "¿sobrevivió: sí o no?".

## 🧭 Lo que vas a aprender
1. Por qué una regresión lineal **no sirve** para predecir una categoría, y qué hace distinto la **logística**
2. Imputar datos faltantes sin filtrar información del conjunto de prueba
3. Interpretar coeficientes como **razón de momios** (*odds ratio*)
4. Evaluar un clasificador: **matriz de confusión, precisión, recall y F1** — y por qué la exactitud engaña
5. Mover el **umbral de decisión** y leer la **curva ROC**

## ⏱️ Agenda (3 horas)
| Parte | Tema | Tiempo |
|---|---|---|
| 0 | Preparación | 10 min |
| 1 | Conocer los datos | 20 min |
| 2 | Explorar: ¿quién sobrevivió? | 20 min |
| 3 | De la regresión lineal a la **logística** | 20 min |
| ☕ | Descanso | 10 min |
| 4 | Preprocesamiento: imputar y preparar variables | 20 min |
| 5 | Ajustar el modelo e **interpretarlo** | 30 min |
| 6 | Evaluar un clasificador | 25 min |
| 7 | El umbral y la curva ROC | 15 min |
| 8 | Conclusiones | 10 min |

## 📋 Las variables
| Variable | Descripción |
|---|---|
| **`survived`** | 🎯 **Variable objetivo:** 1 = sobrevivió, 0 = no sobrevivió |
| `pclass` | Clase del boleto: 1, 2 o 3 |
| `sex` | `male` o `female` |
| `age` | Edad en años |
| `sibsp` | Hermanos o cónyuge a bordo |
| `parch` | Padres o hijos a bordo |
| `fare` | Tarifa pagada (libras esterlinas) |
| `embarked` | Puerto: `C` = Cherburgo, `Q` = Queenstown, `S` = Southampton |
| `adult_male`, `alone` | Columnas ya derivadas por el dataset (ojo con ellas 👀) |

---
# 0️⃣ Preparación

Ejecuta las siguientes celdas **una sola vez**. Traen las funciones que ya conoces de la sesión pasada, más las nuevas de clasificación.

## 🛠️ Funciones auxiliares de visualización

Ejecuta la siguiente celda **una sola vez** al inicio. Después sólo tienes que **llamar** a la función que necesites:

| Función | ¿Para qué sirve? |
|---|---|
| `plot_distributions(df, columnas)` | Histograma + boxplot de variables numéricas |
| `plot_frequencies(df, columnas, top_n=None)` | Frecuencia de variables categóricas |
| `plot_correlation_matrix(df, columnas)` | Matriz de correlación |
| `plot_pairplot(df, columnas, color=None)` | Dispersión entre todas las variables numéricas |
| `plot_simple_regression(x, y, results)` | Recta ajustada de un modelo OLS con 1 variable |
| `plot_actual_vs_predicted(y_real, y_pred)` | Valores reales vs predichos |
| `plot_residuals(y_real, y_pred)` | Residuales vs predichos |
| `plot_rfecv(rfecv)` | R² según el número de variables seleccionadas por RFECV |

In [ ]:
# Funciones auxiliares de visualización
# Ejecuta esta celda una vez; después sólo llama a las funciones.
import numpy as np
import plotly.express as px
import plotly.graph_objects as go


def plot_distributions(df, columns, nbins=30):
    """Histograma con boxplot marginal para cada variable numérica."""
    for col in columns:
        fig = px.histogram(
            df,
            x=col,
            nbins=nbins,
            marginal='box',
            opacity=0.7,
            title=f'Distribución de {col}'
        )
        fig.update_layout(bargap=0.2)
        fig.show()


def plot_frequencies(df, columns, top_n=None):
    """Gráfica de barras con la frecuencia de cada categoría (top_n limita a las más comunes)."""
    for col in columns:
        freq = df[col].value_counts()
        if top_n:
            freq = freq.head(top_n)
        freq_df = freq.rename_axis(col).reset_index(name='Frecuencia')

        title = f'Frecuencias de {col}'
        if top_n and df[col].nunique() > top_n:
            title += f' (top {top_n})'

        fig = px.bar(freq_df, x=col, y='Frecuencia', title=title)
        fig.update_layout(xaxis={'categoryorder': 'total descending'})
        fig.show()


def plot_correlation_matrix(df, columns):
    """Mapa de calor con la correlación de Pearson entre las variables numéricas."""
    corr = df[columns].corr().round(2)
    fig = px.imshow(
        corr,
        text_auto=True,
        color_continuous_scale='RdBu_r',
        zmin=-1,
        zmax=1,
        title='Matriz de Correlación'
    )
    fig.update_layout(width=750, height=650)
    fig.show()


def plot_pairplot(df, columns, color=None):
    """Matriz de dispersión (pairplot) entre las variables numéricas."""
    fig = px.scatter_matrix(
        df,
        dimensions=columns,
        color=color,
        title='Pairplot de Variables Numéricas',
        labels={col: col.capitalize() for col in columns}
    )
    fig.update_layout(width=1200, height=1200, title_font_size=20)
    fig.update_traces(diagonal_visible=True)
    fig.show()


def plot_simple_regression(x, y, results):
    """Dispersión de una variable vs el objetivo con la recta ajustada por un OLS de 1 variable."""
    b0, b1 = results.params.iloc[0], results.params.iloc[1]
    x_name = getattr(x, 'name', None) or 'x'
    y_name = getattr(y, 'name', None) or 'y'
    x_line = np.linspace(np.min(x), np.max(x), 100)

    fig = px.scatter(
        x=np.asarray(x),
        y=np.asarray(y),
        opacity=0.6,
        labels={'x': x_name, 'y': y_name},
        title=f'{y_name} = {b0:.2f} + ({b1:.4f}) · {x_name}',
        template='plotly_white'
    )
    fig.add_trace(go.Scatter(
        x=x_line,
        y=b0 + b1 * x_line,
        mode='lines',
        name='Recta OLS',
        line=dict(color='red', width=3)
    ))
    fig.show()


def plot_actual_vs_predicted(y_true, y_pred, title='Real vs Predicho'):
    """Valores reales vs predichos; un modelo perfecto cae sobre la diagonal roja."""
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    lo = min(y_true.min(), y_pred.min())
    hi = max(y_true.max(), y_pred.max())

    fig = px.scatter(
        x=y_true,
        y=y_pred,
        opacity=0.5,
        labels={'x': 'Valor real', 'y': 'Valor predicho'},
        title=title,
        template='plotly_white'
    )
    fig.add_shape(
        type='line', x0=lo, y0=lo, x1=hi, y1=hi,
        line=dict(color='red', dash='dash')
    )
    fig.show()


def plot_residuals(y_true, y_pred, title='Residuales vs Predicho'):
    """Residuales vs predichos; buscamos una nube sin patrón alrededor de 0."""
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)

    fig = px.scatter(
        x=y_pred,
        y=y_true - y_pred,
        opacity=0.5,
        labels={'x': 'Valor predicho', 'y': 'Residual (real − predicho)'},
        title=title,
        template='plotly_white'
    )
    fig.add_hline(y=0, line_dash='dash', line_color='red')
    fig.show()


def plot_rfecv(rfecv):
    """R² promedio de validación cruzada según el número de variables que conserva RFECV."""
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=rfecv.cv_results_['n_features'],
        y=rfecv.cv_results_['mean_test_score'],
        mode='lines+markers',
        line=dict(color='steelblue', width=3),
        marker=dict(size=7),
        name='R² promedio (CV)'
    ))
    fig.update_layout(
        title='RFECV — R² según número de variables seleccionadas',
        xaxis_title='Número de variables',
        yaxis_title='R² (validación cruzada)',
        template='plotly_white',
        width=900, height=450
    )
    fig.show()

### 🧰 Funciones de clasificación

Ejecuta la celda una vez; después sólo llamas a la función que necesites:

| Función | ¿Para qué sirve? |
|---|---|
| `ajustar_logit(X_train, y_train)` | Ajusta una regresión logística (ya agrega la constante) |
| `predecir_proba(results, X)` | Probabilidad estimada de la clase positiva |
| `razon_de_momios(results)` | Coeficientes, razón de momios y p-value en una tabla |
| `evaluar_clasificador(nombre, y_real, y_pred)` | Exactitud, precisión, recall y F1 |
| `plot_tasa_por_categoria(df, columna, objetivo)` | Tasa de la clase positiva por categoría |
| `plot_matriz_confusion(y_real, y_pred, etiquetas)` | Matriz de confusión |
| `plot_curva_roc(y_real, y_proba)` | Curva ROC y AUC |
| `plot_metricas_por_umbral(y_real, y_proba)` | Cómo cambian las métricas al mover el umbral |
| `plot_sigmoide()` | La curva que convierte cualquier número en probabilidad |

In [ ]:
# Funciones de clasificación: ajustar, evaluar y diagnosticar modelos de clasificación
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import statsmodels.api as sm
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score,
                             precision_score, recall_score, roc_auc_score, roc_curve)


def ajustar_logit(X_train, y_train):
    """Ajusta una regresión logística (agrega la constante automáticamente)."""
    return sm.Logit(y_train, sm.add_constant(X_train)).fit(disp=0)


def predecir_proba(results, X):
    """Probabilidad estimada de la clase positiva, entre 0 y 1."""
    return results.predict(sm.add_constant(X))


def razon_de_momios(results):
    """Coeficientes en log-odds, su razón de momios (odds ratio) y el p-value."""
    return pd.DataFrame({
        'coef (log-odds)': results.params.round(4),
        'razón de momios': np.exp(results.params).round(3),
        'p-value': results.pvalues.round(4),
    })


def evaluar_clasificador(nombre, y_true, y_pred):
    """Imprime y regresa exactitud, precisión, recall y F1."""
    metricas = {
        'modelo': nombre,
        'exactitud': round(accuracy_score(y_true, y_pred), 3),
        'precisión': round(precision_score(y_true, y_pred, zero_division=0), 3),
        'recall': round(recall_score(y_true, y_pred, zero_division=0), 3),
        'F1': round(f1_score(y_true, y_pred, zero_division=0), 3),
    }
    print(f"{nombre}: exactitud {metricas['exactitud']} | precisión {metricas['precisión']} "
          f"| recall {metricas['recall']} | F1 {metricas['F1']}")
    return metricas


def plot_tasa_por_categoria(df, columna, objetivo):
    """Proporción de la clase positiva dentro de cada categoría."""
    tasa = df.groupby(columna, as_index=False)[objetivo].mean()

    fig = px.bar(
        tasa,
        x=columna,
        y=objetivo,
        text_auto='.1%',
        title=f'Tasa de {objetivo} por {columna}',
        template='plotly_white'
    )
    fig.update_layout(yaxis_tickformat='.0%', yaxis_title=f'Tasa de {objetivo}', width=700, height=450)
    fig.update_xaxes(type='category')
    fig.show()


def plot_matriz_confusion(y_true, y_pred, etiquetas=('No', 'Sí'), title='Matriz de confusión'):
    """Aciertos y errores del clasificador: la diagonal son los aciertos."""
    cm = pd.DataFrame(
        confusion_matrix(y_true, y_pred),
        index=[f'Real: {e}' for e in etiquetas],
        columns=[f'Predicho: {e}' for e in etiquetas]
    )

    fig = px.imshow(cm, text_auto=True, color_continuous_scale='Blues', title=title)
    fig.update_layout(width=650, height=450)
    fig.show()


def plot_curva_roc(y_true, y_proba, title='Curva ROC'):
    """Compromiso entre verdaderos y falsos positivos al mover el umbral."""
    fpr, tpr, _ = roc_curve(y_true, y_proba)
    auc = roc_auc_score(y_true, y_proba)

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=fpr, y=tpr, mode='lines', name=f'Modelo (AUC = {auc:.3f})',
        line=dict(color='steelblue', width=3)
    ))
    fig.add_trace(go.Scatter(
        x=[0, 1], y=[0, 1], mode='lines', name='Azar (AUC = 0.5)',
        line=dict(color='red', dash='dash')
    ))
    fig.update_layout(
        title=title,
        xaxis_title='Falsos positivos (1 − especificidad)',
        yaxis_title='Verdaderos positivos (recall)',
        template='plotly_white', width=700, height=550
    )
    fig.show()


def plot_metricas_por_umbral(y_true, y_proba):
    """Cómo cambian exactitud, precisión, recall y F1 según el umbral de decisión."""
    umbrales = np.arange(0.05, 0.96, 0.05)
    filas = []
    for u in umbrales:
        y_pred = (y_proba >= u).astype(int)
        filas.append({
            'umbral': round(u, 2),
            'exactitud': accuracy_score(y_true, y_pred),
            'precisión': precision_score(y_true, y_pred, zero_division=0),
            'recall': recall_score(y_true, y_pred, zero_division=0),
            'F1': f1_score(y_true, y_pred, zero_division=0),
        })

    datos = pd.DataFrame(filas).melt(id_vars='umbral', var_name='métrica', value_name='valor')

    fig = px.line(
        datos, x='umbral', y='valor', color='métrica', markers=True,
        title='Métricas según el umbral de decisión', template='plotly_white'
    )
    fig.add_vline(x=0.5, line_dash='dash', line_color='gray', annotation_text='umbral por defecto')
    fig.update_layout(width=900, height=500)
    fig.show()


def plot_sigmoide():
    """La función que convierte cualquier número en una probabilidad entre 0 y 1."""
    z = np.linspace(-8, 8, 200)

    fig = px.line(
        x=z, y=1 / (1 + np.exp(-z)),
        labels={'x': 'z = β₀ + β₁x₁ + β₂x₂ + …', 'y': 'Probabilidad estimada'},
        title='Función sigmoide (logística)', template='plotly_white'
    )
    fig.add_hline(y=0.5, line_dash='dash', line_color='red', annotation_text='umbral 0.5')
    fig.add_vline(x=0, line_dash='dot', line_color='gray')
    fig.update_traces(line=dict(color='steelblue', width=3))
    fig.update_layout(width=800, height=450)
    fig.show()

### 📥 Cargar los datos
Misma fuente que la sesión anterior: una base **SQLite** que descargamos y consultamos con SQL.

In [ ]:
import requests, sqlite3, pandas as pd

url = "https://raw.githubusercontent.com/davidjamesknight/SQLite_databases_for_learning_data_science/main/titanic.db"
r = requests.get(url)

with open("titanic.db", "wb") as f:
    f.write(r.content)

conn = sqlite3.connect("titanic.db")

query = """
SELECT
    O.survived,
    O.pclass,
    O.age,
    O.sibsp,
    O.parch,
    O.fare,
    O.adult_male,
    O.alone,
    S.sex,
    E.embarked
FROM
    Observation AS O
JOIN
    Sex AS S ON O.sex_id = S.sex_id
JOIN
    Embarked AS E ON O.embarked_id = E.embarked_id
"""

df = pd.read_sql_query(query, conn)
df.head()

---
# 1️⃣ Conocer los datos (20 min)

Tres preguntas de siempre: ¿cuántos datos hay?, ¿faltan valores?, ¿cómo está repartida la variable objetivo?

In [ ]:
# Dimensiones y tipos de datos
print(f"Filas: {df.shape[0]}, columnas: {df.shape[1]}")
display(df.dtypes)

In [ ]:
# Valores nulos por columna
df.isnull().sum()

### Aquí **no** podemos eliminar las filas con nulos

En la sesión pasada eliminamos los nulos de `horsepower` porque eran el **1.5%** de las filas. Calcula el porcentaje aquí antes de decidir.

In [ ]:
# Porcentaje de valores nulos en las columnas indicadas
(df[['age', 'embarked']].isnull().mean() * 100).round(2)

✍️ **¿Qué observas?**

- age tiene **19.87%** de valores nulos (177 de 891). Eliminar esas filas haría perder casi una quinta parte de la muestra, así que es mejor imputar la edad.
- embarked sólo tiene **0.22%** de nulos (2 filas). Se podrían eliminar sin afectar mucho el análisis, pero aquí los imputaremos con la moda calculada únicamente en el conjunto de entrenamiento para mantener un procedimiento consistente y evitar fuga de datos.

### ¿Cómo está repartida la variable objetivo?
En clasificación esto es lo primero que se revisa: si el 99% de los casos fueran de una sola clase, un modelo que siempre responda esa clase acertaría el 99% de las veces sin aprender nada.

In [ ]:
# Proporción de pasajeros que sobrevivieron y no sobrevivieron
df['survived'].value_counts(normalize=True).sort_index()

✍️ Sobrevivió aproximadamente el **38.38%** de los pasajeros; el **61.62%** no sobrevivió. Por ello, un clasificador que siempre predijera “no sobrevivió” tendría cerca de 61.6% de exactitud sin aprender nada útil.

---
# 2️⃣ Explorar: ¿quién sobrevivió? (20 min)

Con una variable objetivo de 0 y 1, el gráfico más útil es la **tasa de supervivencia por grupo**: el promedio de `survived` dentro de cada categoría.

In [ ]:
# Tasa de supervivencia por sexo
plot_tasa_por_categoria(df, 'sex', 'survived')

In [ ]:
# Tasa de supervivencia por clase
plot_tasa_por_categoria(df, 'pclass', 'survived')

In [ ]:
# Tasa de supervivencia por puerto de embarque
plot_tasa_por_categoria(df, 'embarked', 'survived')

### ¿Y la edad?
`age` es numérica, así que la agrupamos en rangos para poder ver la tasa por grupo.

In [ ]:
# Esta celda va de regalo: léela con calma y ejecútala
df_edad = df.assign(
    grupo_edad=pd.cut(df['age'], [0, 16, 30, 50, 100], labels=['0-16', '17-30', '31-50', '51+'])
)

plot_tasa_por_categoria(df_edad, 'grupo_edad', 'survived')

✍️ **¿Qué observas?**

- El **sexo** parece ser la variable con mayor influencia: sobrevivió cerca del **74.2% de las mujeres**, frente al **18.9% de los hombres**.
- El grupo de **0 a 16 años** tuvo la mayor supervivencia, aproximadamente **55.0%**.
- Cherburgo presenta una tasa mayor que Southampton (55.4% frente a 33.7%), pero no significa que el puerto salvara vidas. Hay una variable de confusión: en Cherburgo, aproximadamente **50.6%** de los pasajeros viajaba en primera clase, mientras que en Southampton sólo **19.7%** lo hacía. La clase del boleto explica parte importante de esa diferencia.

---
# 3️⃣ De la regresión lineal a la logística (20 min)

### ¿Por qué no usar la regresión lineal de la sesión pasada?

Imagina ajustar una recta a `survived` (que sólo vale 0 o 1) usando la edad:

$$\text{survived} = \beta_0 + \beta_1 \cdot \text{age}$$

Una recta **no tiene límites**: para edades altas predeciría valores negativos, y para otras, valores mayores a 1. **¿Qué significa una supervivencia de -0.3 o de 1.4?** Nada.

### La solución: la función sigmoide

La regresión logística toma la misma combinación lineal de siempre…

$$z = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \dots$$

…y la pasa por una función que **aplasta cualquier número al rango de 0 a 1**:

$$P(\text{sobrevivir}) = \frac{1}{1 + e^{-z}}$$

In [ ]:
# Esta celda va de regalo: léela con calma y ejecútala
plot_sigmoide()

✅ **Qué observar en la curva:**
- Si $z = 0$, la probabilidad es exactamente **0.5**
- Valores muy negativos de $z$ → probabilidad cercana a **0**; muy positivos → cercana a **1**
- Nunca se sale del rango 0 a 1, sin importar qué tan grande sea $z$

### Lo que cambia y lo que no

| | Regresión lineal | Regresión logística |
|---|---|---|
| Predice | Un número cualquiera | Una **probabilidad** (0 a 1) |
| Ecuación | $y = \beta_0 + \beta_1x$ | $P = \dfrac{1}{1+e^{-(\beta_0 + \beta_1x)}}$ |
| Coeficientes | Cambio en $y$ por unidad de $x$ | Cambio en el **log-odds** por unidad de $x$ |
| Se ajusta con | Mínimos cuadrados (OLS) | Máxima verosimilitud |
| Métricas | R², RMSE | Exactitud, precisión, recall, F1 |

Lo que **no** cambia: train/test, el papel de los p-values, y el cuidado con las variables redundantes.

---
# ☕ Descanso (10 min)

---
# 4️⃣ Preprocesamiento (20 min)

### Primero dividir, después imputar
El orden importa. Si imputamos la edad con la mediana de **todos** los datos y luego dividimos, la mediana del conjunto de prueba se habrá colado en el entrenamiento. Eso se llama **fuga de datos** (*data leakage*) y hace que el modelo se vea mejor de lo que es.

Usamos `stratify=y` para que train y test conserven la misma proporción de sobrevivientes.

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=['survived', 'adult_male', 'alone'])
y = df['survived']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print('X_train:', X_train.shape, '| X_test:', X_test.shape)
print('Tasa de supervivencia en train:', round(y_train.mean(), 3))
print('Tasa de supervivencia en test:', round(y_test.mean(), 3))

### Imputar los faltantes
- `age` → la **mediana** (es robusta a valores extremos)
- `embarked` → la **moda**, es decir, el puerto más frecuente

Ambos valores se calculan **sólo con train** y se aplican a los dos conjuntos.

> ⚠️ **Recuerda (pandas 3):** `df['col'].fillna(valor, inplace=True)` ya no modifica nada. Usa `df['col'] = df['col'].fillna(valor)`.

In [ ]:
mediana_edad = X_train['age'].median()
moda_puerto = X_train['embarked'].mode()[0]

X_train_prep = X_train.copy()
X_test_prep = X_test.copy()

for datos in (X_train_prep, X_test_prep):
    datos['age'] = datos['age'].fillna(mediana_edad)
    datos['embarked'] = datos['embarked'].fillna(moda_puerto)

print('Mediana de edad (train):', mediana_edad)
print('Moda del puerto (train):', moda_puerto)
print('Nulos en train:', X_train_prep.isnull().sum().sum())
print('Nulos en test:', X_test_prep.isnull().sum().sum())

### Crear variables nuevas (*feature engineering*)
A veces una variable útil no viene en los datos, pero se puede construir:
- **`viaja_solo`**: 1 si el pasajero no lleva familiares a bordo (`sibsp + parch == 0`)
- **`es_menor`**: 1 si tiene 16 años o menos

`np.where(condición, valor_si_verdadero, valor_si_falso)` es la forma directa de hacerlo.

In [ ]:
import numpy as np

for datos in (X_train_prep, X_test_prep):
    datos['viaja_solo'] = np.where((datos['sibsp'] + datos['parch']) == 0, 1, 0)
    datos['es_menor'] = np.where(datos['age'] <= 16, 1, 0)
    datos.drop(columns=['sibsp', 'parch'], inplace=True)

X_train_prep.head()

### Variables categóricas a dummies
Igual que con `origin` en la sesión pasada: una columna menos que categorías, y la categoría que quitamos se vuelve la **referencia** contra la que se comparan las demás.

Nuestras referencias serán: **3ª clase**, puerto **Southampton** y **mujer**. Así cada coeficiente se leerá como *"comparado con una mujer de tercera clase que embarcó en Southampton"*.

In [ ]:
# Esta celda va de regalo: léela con calma y ejecútala
from sklearn.preprocessing import OneHotEncoder

categoricas = ['pclass', 'embarked', 'sex']

ohe = OneHotEncoder(drop=[3, 'S', 'female'], sparse_output=False)

dummies_train = pd.DataFrame(
    ohe.fit_transform(X_train_prep[categoricas]),
    columns=ohe.get_feature_names_out(),
    index=X_train_prep.index
)
dummies_test = pd.DataFrame(
    ohe.transform(X_test_prep[categoricas]),
    columns=ohe.get_feature_names_out(),
    index=X_test_prep.index
)

X_train_enc = pd.concat([X_train_prep.drop(columns=categoricas), dummies_train], axis=1)
X_test_enc = pd.concat([X_test_prep.drop(columns=categoricas), dummies_test], axis=1)

X_train_enc.head()

---
# 5️⃣ Ajustar el modelo e interpretarlo (30 min)

Usamos `statsmodels`, igual que en la sesión pasada, para tener el resumen estadístico completo.

In [ ]:
modelo_completo = ajustar_logit(X_train_enc, y_train)
print(modelo_completo.summary())

### 🔍 Cómo leer este resumen
Es casi igual al de la sesión pasada, con dos diferencias:

| Dónde | Qué es |
|---|---|
| `Pseudo R-squ.` | Parecido al R², pero no se interpreta como "% explicado". Sirve para comparar modelos del mismo tipo |
| Columna `coef` | El cambio en el **log-odds**, no en la probabilidad. Por eso los traducimos abajo |
| Columna `P>\|z\|` | El p-value de siempre: < 0.05 → la variable es significativa |

### De log-odds a razón de momios
Un coeficiente de `-2.62` no dice gran cosa. Pero si le aplicamos $e^{-2.62} = 0.073$, ya se puede leer: **los momios de sobrevivir se multiplican por 0.073**, o sea que caen un 93%.

**Momios** (*odds*) = probabilidad de que pase entre probabilidad de que no pase. Si 3 de cada 4 sobreviven, los momios son 3 a 1.

In [ ]:
# Coeficientes, razones de momios y significancia estadística
razon_de_momios(modelo_completo)

✍️ **Interpreta tus resultados:**

- La variable con el efecto más fuerte es sex_male: su coeficiente es aproximadamente **−2.62** y su razón de momios es **0.073**. Manteniendo las demás variables constantes, los momios de sobrevivir de un hombre son alrededor de **92.7% menores** que los de una mujer.
- Viajar en primera clase (pclass_1) multiplica los momios de sobrevivir por aproximadamente **10.39** frente a viajar en tercera clase.
- Entre los predictores, fare, viaja_solo, es_menor y embarked_C tienen p-value mayor a 0.05 en el modelo completo. fare no aporta evidencia adicional porque está relacionada con la clase del boleto (correlación aproximada de **−0.55** con pclass); una vez incluida la clase, gran parte de la información de la tarifa ya está representada.

### 👀 La trampa que dejamos pendiente

Al inicio quitamos `alone` y `adult_male` porque el dataset ya las traía derivadas. Resulta que **`alone` es exactamente la misma columna que la `viaja_solo` que acabas de construir**. Veamos qué pasa si el modelo recibe las dos.

In [ ]:
# Esta celda va de regalo: léela con calma y ejecútala
alone_original = np.where(X_train['sibsp'] + X_train['parch'] > 0, 0, 1)
print("¿'alone' y 'viaja_solo' son idénticas?",
      (alone_original == X_train_enc['viaja_solo']).all())

X_con_duplicada = X_train_enc.assign(alone=alone_original)

try:
    ajustar_logit(X_con_duplicada, y_train)
except Exception as error:
    print(f"\n💥 {type(error).__name__}: {error}")

✍️ Aparece el error **LinAlgError: Singular matrix**. alone y viaja_solo contienen exactamente la misma información, por lo que existe multicolinealidad perfecta. La matriz del modelo no se puede invertir y los coeficientes no se pueden estimar por separado. Es el caso extremo de un VIF muy alto (en teoría, infinito): una variable se explica completamente mediante otra.

---
# 6️⃣ Evaluar un clasificador (25 min)

El modelo devuelve **probabilidades**. Para tomar una decisión hay que fijar un **umbral**: por defecto, probabilidad ≥ 0.5 → predecimos "sobrevivió".

In [ ]:
proba_test = predecir_proba(modelo_completo, X_test_enc)
pred_test = (proba_test >= 0.5).astype(int)

proba_test.head().round(3)

In [ ]:
# Guardaremos las métricas de cada modelo para compararlos al final
comparacion = []

comparacion.append(
    evaluar_clasificador('Logística (umbral 0.5)', y_test, pred_test)
)

### La matriz de confusión
Cuatro casillas: los aciertos están en la diagonal.

| | Predicho: No | Predicho: Sí |
|---|---|---|
| **Real: No** | Verdadero negativo ✅ | Falso positivo ❌ |
| **Real: Sí** | Falso negativo ❌ | Verdadero positivo ✅ |

In [ ]:
plot_matriz_confusion(
    y_test,
    pred_test,
    etiquetas=('No sobrevivió', 'Sobrevivió')
)

✍️ **Lee tu matriz de confusión:**

- El modelo clasificó correctamente a **138 pasajeros**: 95 verdaderos negativos y 43 verdaderos positivos.
- Hubo **26 falsos negativos**: pasajeros que sí sobrevivieron, pero que el modelo clasificó como no sobrevivientes.
- La **precisión de 74.1%** significa que, entre quienes el modelo predijo que sobrevivirían, 74.1% realmente sobrevivió. El **recall de 62.3%** significa que el modelo detectó a 62.3% de todos los sobrevivientes reales.

### ¿Contra qué comparamos? Dos modelos tontos

Un número suelto no dice si el modelo es bueno. Necesitamos puntos de comparación:
1. **"Todos mueren":** predecir siempre 0, sin mirar nada
2. **"Mujeres y niños primero":** la regla histórica, en su versión más simple — *sobreviven las mujeres*

In [ ]:
pred_todos_mueren = np.zeros(len(y_test), dtype=int)
pred_regla_mujeres = (X_test['sex'] == 'female').astype(int)

comparacion.append(
    evaluar_clasificador('Todos mueren', y_test, pred_todos_mueren)
)
comparacion.append(
    evaluar_clasificador('Regla: sobreviven las mujeres', y_test, pred_regla_mujeres)
)

✍️ **Compara los tres resultados:**

- “Todos mueren” logra una exactitud de **61.5%**, pero su recall es **0%** porque no identifica a un solo sobreviviente. Esto demuestra que la exactitud aislada puede engañar cuando las clases no están perfectamente equilibradas.
- La regla de las mujeres obtiene **77.7%** de exactitud y F1 de **0.692**, ligeramente mejor que la logística con umbral 0.5 (77.1% y F1 de 0.677). Sin embargo, el modelo produce probabilidades individuales, combina varias características y permite cambiar el umbral según el costo de los falsos positivos y falsos negativos; la regla simple sólo entrega una decisión fija.

---
# 7️⃣ El umbral y la curva ROC (15 min)

La regla simple da un **sí o no** y ahí se acaba. El modelo da una **probabilidad**, y eso permite algo que la regla no puede: **elegir qué error prefieres cometer**.

El 0.5 no es sagrado. Bajarlo hace al modelo más "optimista" (detecta más sobrevivientes, pero se equivoca más); subirlo lo vuelve más exigente.

In [ ]:
# Cambio de las métricas al mover el umbral
plot_metricas_por_umbral(y_test, proba_test)

In [ ]:
pred_umbral_04 = (proba_test >= 0.4).astype(int)

comparacion.append(
    evaluar_clasificador('Logística (umbral 0.4)', y_test, pred_umbral_04)
)

✍️ Entre los umbrales probados de 0.05 a 0.95, el mejor F1 se obtiene con **0.40**: F1 ≈ **0.731**, precisión ≈ **0.697** y recall ≈ **0.768**. Al bajar el umbral, más casos se clasifican como sobrevivientes; por eso se detecta una proporción mayor de los sobrevivientes reales y aumenta el recall, aunque normalmente también aumentan los falsos positivos.

### La curva ROC: el modelo sin comprometerse con un umbral
La curva ROC recorre **todos los umbrales posibles** y grafica cuántos sobrevivientes detectas (eje Y) contra cuántas falsas alarmas generas (eje X).

El **AUC** (área bajo la curva) resume esa curva en un número: la probabilidad de que, tomando un sobreviviente y un no sobreviviente al azar, el modelo le asigne mayor probabilidad al que sí sobrevivió. 0.5 es azar puro, 1.0 es perfecto.

In [ ]:
plot_curva_roc(y_test, proba_test)

✍️ El AUC obtenido es aproximadamente **0.837**. Esto indica una buena capacidad de discriminación: en cerca de 83.7% de las parejas formadas por un sobreviviente y un no sobreviviente, el modelo asigna una probabilidad mayor al sobreviviente. A diferencia de la regla simple, el modelo entrega una clasificación ordenada por riesgo y permite elegir el umbral según qué tipo de error sea más costoso.

---
# 8️⃣ Conclusiones (10 min)

Cerramos quitando las dos variables que no eran significativas (`fare` y `viaja_solo`), como hicimos en la sesión pasada con los p-values.

In [ ]:
columnas_reducido = X_train_enc.columns.drop(['fare', 'viaja_solo'])

modelo_reducido = ajustar_logit(X_train_enc[columnas_reducido], y_train)
proba_reducido = predecir_proba(modelo_reducido, X_test_enc[columnas_reducido])
pred_reducido = (proba_reducido >= 0.5).astype(int)

comparacion.append(
    evaluar_clasificador('Logística reducida (umbral 0.5)', y_test, pred_reducido)
)

razon_de_momios(modelo_reducido)

In [ ]:
# Tabla comparativa de todos los modelos probados
pd.DataFrame(comparacion).sort_values('F1', ascending=False).reset_index(drop=True)

✍️ **Responde la pregunta del inicio:** ¿quién sobrevivió al Titanic?

Los resultados indican que las probabilidades de sobrevivir fueron mucho mayores para **las mujeres, los pasajeros de primera y segunda clase y, en cierta medida, los pasajeros más jóvenes**. En el modelo reducido, ser hombre multiplica los momios de sobrevivir por sólo **0.076** frente a ser mujer (una reducción cercana al 92.4%), mientras que viajar en primera clase los multiplica por aproximadamente **10.75** frente a tercera clase. Cada año adicional de edad multiplica los momios por cerca de **0.975**, equivalente a una disminución aproximada de 2.5% por año, manteniendo las demás variables constantes.

Sí podemos predecir la supervivencia con estas características, aunque no con certeza. El modelo alcanza un AUC aproximado de **0.837** y permite producir probabilidades individuales y ajustar el umbral según el tipo de error que queramos reducir.

## 📝 Lo que aprendimos

1. En clasificación no basta con revisar la exactitud: también hay que interpretar precisión, recall, F1, matriz de confusión y AUC.
2. La imputación debe calcularse sólo con el conjunto de entrenamiento para evitar fuga de datos.
3. Los coeficientes logísticos se interpretan mejor mediante razones de momios, y las variables duplicadas producen multicolinealidad perfecta.

## 🚀 Retos opcionales
- Con plot_metricas_por_umbral busca el umbral que maximiza el F1 y evalúa ese modelo. ¿Le gana al de umbral 0.4?
- Agrega adult_male al modelo. ¿Qué le pasa a los p-values de sex_male y es_menor? ¿Por qué?
- Ajusta la logística usando sólo sex_male. ¿Qué exactitud da? Compárala con la regla “las mujeres sobreviven” y explica el resultado.